<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# Solution: Set up PyTorch

**[Set up PyTorch](https://learning.nextia-ai.com/courses/deep-learning/m02/set-up-pytorch/)** · Deep Learning and Transformers Explained · Module 2, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** check that PyTorch works, run the course's known-good first run and compare it with the lesson, number by number, and download Larkfield's tickets.

| | |
|---|---|
| **Time** | About 10 minutes |
| **Needs** | A browser and a free Colab or Kaggle account. PyTorch is already installed there. In Kaggle, turn on Internet in the notebook settings for the data cell. |
| **You should know** | Tensors, shapes and dtypes, from [Tensors and shapes](https://learning.nextia-ai.com/courses/deep-learning/m02/tensors-and-shapes/). |
| **Tested** | Python 3.12 with PyTorch 2.14.1 (CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have PyTorch preinstalled |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m02/set-up-pytorch/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M02-L02-set-up-pytorch/set-up-pytorch-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. 

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: get the tickets and the course's code

Run the next cell. It downloads Larkfield's ticket texts (about 2 MB) and the course's `ticketnet.py` into a folder `ticket-router`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 4696 tickets, valid 600, test 600.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `ticket-router` (in Colab: the **Files** panel on the left), then run the cell again.

The tickets are made up for this course by a language model, from a list of situations; no real customer wrote them.

In [ ]:
# Setup: download the tickets and ticketnet.py into ticket-router/ and check them.
import hashlib, os, subprocess, sys, urllib.request
from pathlib import Path

LABS = os.environ.get("NX_LABS_RAW", "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C06/")
FILES = {
    "train.csv": "98487169346f8b0d9bc55a3844849ab95d012b75f01d7083605a6a1cea57aa39",
    "valid.csv": "f0c7bfa75e75837cdaf891ac8d007a78a018ba8e3488155cf43f2c6a34a064c8",
    "test.csv": "f373f402bcc9742cdcf8829720710a97b7a1dce5d8e34aaa88d03e7e49a5404e",
    "split_manifest.json": "d043cf720697054fb16e6cfff72ecb49b471d75038308ad5836779a0dd0bf8a5",
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-router").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:  # Python without certificates (python.org on macOS): try curl
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-router and run again.")
download(LABS + "project/ticketnet.py", PROJECT / "ticketnet.py")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
import ticketnet
from ticketnet import *  # TEAMS, Vocab, tokenize, the three networks, train(), evaluate() ...
train_rows, valid_rows, test_rows = (load_split(f"data/{p}.csv") for p in ("train", "valid", "test"))
print(f"Ready: train {len(train_rows)} tickets, valid {len(valid_rows)}, test {len(test_rows)}.")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The known-good first run

This is the program `first_run.py` of the lesson. It prints the versions, a tensor of random numbers made with **seed** 0, its sum, and the accelerator that it finds.

> **Predict.** The lesson shows the tensor `[[0.4963, 0.7682, 0.0885], [0.1320, 0.3074, 0.6341]]`. Will yours be the same? Then run the cell.

In [ ]:
import platform

import torch

print("Python ", platform.python_version())
print("PyTorch", torch.__version__)

torch.manual_seed(0)              # the same seed gives the same random numbers
x = torch.rand(2, 3)              # 2 rows, 3 columns, numbers between 0 and 1
print(x)
print("shape", tuple(x.shape), "| dtype", x.dtype, "| device", x.device)
print("sum", round(x.sum().item(), 4))

if torch.cuda.is_available():
    accelerator = "cuda (an NVIDIA graphics card)"
elif torch.backends.mps.is_available():
    accelerator = "mps (an Apple silicon graphics unit)"
else:
    accelerator = "none"
print("accelerator:", accelerator, "| this course uses the CPU")

> **Check.** The tensor, the line `shape (2, 3) | dtype torch.float32 | device cpu` and `sum 2.4265` must be the same as in the lesson. The Python and PyTorch versions can differ. In a Colab or Kaggle CPU session, the accelerator line says `none`.

The next cell checks the sum for you.

In [ ]:
expect('the sum of the seeded tensor', round(x.sum().item(), 4), 2.4265)

## 2. Without a seed, and with a seed

> **Predict.** The first cell calls `torch.rand(3)` twice without a seed. The second sets seed 0 before each call. Which cell prints the same numbers twice? Run both.

In [ ]:
print(torch.rand(3))
print(torch.rand(3))

In [ ]:
torch.manual_seed(0)
print(torch.rand(3))
torch.manual_seed(0)
print(torch.rand(3))

> **Check.** The first cell prints two different rows, and they differ from the lesson's. The second prints `tensor([0.4963, 0.7682, 0.0885])` twice.

The seed also fixes the starting weights of a new layer. Run the cell.

In [ ]:
from torch import nn

torch.manual_seed(0)
layer = nn.Linear(3, 2)     # a new layer: 3 inputs, 2 outputs, random starting weights
print(layer.weight)

> **Check.** You should see `[[-0.0043,  0.3097, -0.4752], [-0.4249, -0.2224,  0.1548]]`.

## 3. Change one thing: another seed

> **Your turn.** Set the seed to 1, make `x1 = torch.rand(2, 3)`, and print it with its sum. Before you run it, predict: is it the same as with seed 0? Then run the check cell.

In [ ]:
torch.manual_seed(1)
x1 = torch.rand(2, 3)
print(x1)
print("sum", round(x1.sum().item(), 4))

In [ ]:
expect('the sum with seed 1', round(x1.sum().item(), 4) if hasattr(x1, "sum") else x1, 3.0038)

Each seed starts its own fixed sequence. This course always uses seed 0.

## 4. The tickets

The setup cell at the top downloaded the tickets and `ticketnet.py`, and checked their checksums. Run the cell to see the first validation ticket.

In [ ]:
print(len(train_rows), "train,", len(valid_rows), "valid,", len(test_rows), "test")
print(valid_rows[0]["ticket_id"], valid_rows[0]["team"])
print(valid_rows[0]["text"][:120], "...")

> **Check.** You should see `4696 train, 600 valid, 600 test` and the ticket `T-64697 payment`, which starts with "I am writing regarding order LK-714258."

## 5. Reset

If something goes wrong: in Colab, select **Runtime › Disconnect and delete runtime**; in Kaggle, stop the session and start it again. Then run every cell from the top. The setup cell downloads the files again if one is missing or changed.

## Next

Your setup is good when the tensor and the sum 2.4265 match the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m02/set-up-pytorch/) also shows the local route on macOS, Windows and Linux, and the knowledge checks. Next, you turn the tickets into tensors and send them through a first network.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [The forward pass](https://learning.nextia-ai.com/courses/deep-learning/m02/the-forward-pass/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m02/set-up-pytorch/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.